In [1]:
import sys
import os

# Add parent directory to path to import from app
sys.path.insert(0, os.path.abspath('..'))

from app.db.core.models.market_data_models import *
from app.db.core.models.user_data_models import *
from app.db.core.models.prophit_alts_models import *
from app.db.core.db_config import *
from app.utils.decorators.database import with_session, with_transaction
from app.utils.serialize_output import serialize_sqlalchemy_obj
import json
import pandas as pd
from app.core.calculations.core.helpers import build_returns_df_for_dates
from datetime import datetime, timedelta
import requests
import os
from dotenv import load_dotenv
from sklearn.decomposition import PCA

# Load API key
load_dotenv()
api_key = os.getenv("FMP_API_KEY")

In [2]:
end_date = datetime.now()
start_date = end_date - timedelta(days=365*3)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']

equity_returns_df = build_returns_df_for_dates(
    tickers,
    start_date,
    end_date,
    include_dividends=False,
    drop_rows='any'
)

def get_fmp_data(url):
    """Helper function to fetch data from FMP API"""
    separator = '&' if '?' in url else '?'
    full_url = f"{url}{separator}apikey={api_key}"
    try:
        response = requests.get(full_url)
        response.raise_for_status()
        data = response.json()
        return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame([data])
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

def get_treasury_rates():
    # Treasury Yields - get and rename to desired format
    treasury_data = get_fmp_data("https://financialmodelingprep.com/api/v4/treasury")

    # Rename columns: month1 -> 1m, year1 -> 1y, etc.
    column_mapping = {
        'month1': '1m', 'month3': '3m', 'month6': '6m',
        'year1': '1y', 'year2': '2y', 'year3': '3y', 'year5': '5y', 'year7': '7y', 'year10': '10y', 'year20': '20y', 'year30': '30y'
    }
    treasury_data = treasury_data.rename(columns=column_mapping)

    # Select only the desired columns
    treasury_rates = treasury_data[['date', '1m', '3m', '6m', '1y', '2y', '3y', '5y', '7y', '10y', '20y', '30y']]

    return treasury_rates

rates_df = get_treasury_rates()


In [3]:
def calc_spreads(df):
    df = df.copy()
    df['10y_2y'] = df['10y'] - df['2y']
    df['10y_3m'] = df['10y'] - df['3m']
    df['5y_2y']  = df['5y'] - df['2y']
    return df[['date', '10y_2y', '10y_3m', '5y_2y']]


In [4]:
def calc_volatility(df, window=20):
    yields = df.drop(columns=['date'])
    vol = yields.rolling(window).std()
    vol['date'] = df['date']
    return vol


In [5]:

def calc_pca(df, n_components=3):
    yields = df.drop(columns=['date']).dropna()
    pca = PCA(n_components=n_components)
    pcs = pca.fit_transform(yields - yields.mean())
    pc_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(n_components)])
    pc_df['date'] = df.loc[yields.index, 'date'].values
    explained = pca.explained_variance_ratio_
    return pc_df, explained


In [6]:
def calc_correlation(df):
    yields = df.drop(columns=['date'])
    changes = yields.diff().dropna()
    return changes.corr()

In [7]:
def calc_daily_changes(df):
    changes = df.drop(columns=['date']).diff()
    changes['date'] = df['date']
    return changes

In [8]:
def calc_inversion_signals(df):
    df = df.copy()
    df['10y_2y'] = df['10y'] - df['2y']
    df['10y_3m'] = df['10y'] - df['3m']
    
    # Boolean flags
    df['inv_10y2y'] = df['10y_2y'] < 0
    df['inv_10y3m'] = df['10y_3m'] < 0
    
    # Persistence (# of inverted days in a row)
    df['inv_10y2y_streak'] = df['inv_10y2y'].groupby((df['inv_10y2y'] != df['inv_10y2y'].shift()).cumsum()).cumsum()
    df['inv_10y3m_streak'] = df['inv_10y3m'].groupby((df['inv_10y3m'] != df['inv_10y3m'].shift()).cumsum()).cumsum()
    
    # Depth
    avg_depth_2y = df.loc[df['10y_2y'] < 0, '10y_2y'].mean()
    avg_depth_3m = df.loc[df['10y_3m'] < 0, '10y_3m'].mean()
    
    return df[['date','10y_2y','10y_3m','inv_10y2y','inv_10y3m','inv_10y2y_streak','inv_10y3m_streak']], {
        "avg_depth_10y2y": avg_depth_2y,
        "avg_depth_10y3m": avg_depth_3m
    }

In [9]:
def calc_term_premium(df):
    short_end = df[['1y','2y']].mean(axis=1)
    long_end  = df[['20y','30y']].mean(axis=1)
    term_premium = long_end - short_end
    return pd.DataFrame({'date': df['date'], 'term_premium': term_premium})

In [23]:
def get_company_profile(ticker: str):
    """
    Retrieves company profile including beta, volume, and other company stats.
    Returns data like: beta, volAvg (average volume), mktCap, lastDiv, etc.
    """
    url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}"
    return get_fmp_data(url)

def get_quote(ticker: str):
    """
    Retrieves full quote including volume and avgVolume.
    """
    url = f"https://financialmodelingprep.com/api/v3/quote/{ticker}"
    return get_fmp_data(url)

# Example usage
profile_df = get_company_profile('AAPL')
quote_df = get_quote('AAPL')

# Merge on 'symbol'
merged_df = profile_df.merge(quote_df, on='symbol', suffixes=('_profile', '_quote'))

print(merged_df.columns)
print(merged_df[['beta', 'isActivelyTrading', 'isAdr', 'isFund', 'ipoDate', 'earningsAnnouncement', 'sharesOutstanding']])



Index(['symbol', 'price_profile', 'beta', 'volAvg', 'mktCap', 'lastDiv',
       'range', 'changes', 'companyName', 'currency', 'cik', 'isin', 'cusip',
       'exchange_profile', 'exchangeShortName', 'industry', 'website',
       'description', 'ceo', 'sector', 'country', 'fullTimeEmployees', 'phone',
       'address', 'city', 'state', 'zip', 'dcfDiff', 'dcf', 'image', 'ipoDate',
       'defaultImage', 'isEtf', 'isActivelyTrading', 'isAdr', 'isFund', 'name',
       'price_quote', 'changesPercentage', 'change', 'dayLow', 'dayHigh',
       'yearHigh', 'yearLow', 'marketCap', 'priceAvg50', 'priceAvg200',
       'exchange_quote', 'volume', 'avgVolume', 'open', 'previousClose', 'eps',
       'pe', 'earningsAnnouncement', 'sharesOutstanding', 'timestamp'],
      dtype='object')
    beta  isActivelyTrading  isAdr  isFund     ipoDate  \
0  1.094               True  False   False  1980-12-12   

           earningsAnnouncement  sharesOutstanding  
0  2025-10-30T20:00:00.000+0000        148403900